In [1]:
from queue import PriorityQueue
from IPython.display import display
import random

In [2]:
class Position:
    def __init__(self,x,y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f'({self.x},{self.y})'


    def __hash__(self):
        return hash((self.x, self.y))

    def __eq__(self, value):
        return self.x == value.x and self.y == value.y


class Cell:
    def __init__(self, position: Position, g = 0,h = 0):
        self.position = position
        self.g = g
        self.h = h
        self.f = g + h

    def __eq__(self, other):
        return self.position == other.position

    def __lt__(self, other):
        return self.f < other.f

    def __hash__(self):
        return hash(self.position)

    def __repr__(self):
        return f'Ô ({self.position.x}, {self.position.y})'


class Grid:
    def __init__(self, rows: int, cols: int):
        self.rows = rows
        self.cols = cols
     
        self.cells = []  
        for x in range(0, self.cols):
            for y in range(0, self.rows):
                self.cells.append(Cell(Position(x,y))) 


    def in_bounds(self, id: Cell):
        x, y  = id.position.x, id.position.y
        return 0 <= x <= self.cols - 1 and 0 <= y <= self.rows - 1

    def neighbors(self, id: Cell):
        x, y  = id.position.x, id.position.y
        neighbors = [Cell(Position(x+1, y)), Cell(Position(x-1, y)), Cell(Position(x, y-1)), Cell(Position(x, y+1)),
                     Cell(Position(x-1,y+1)), Cell(Position(x+1,y+1)), Cell(Position(x-1,y-1)), Cell(Position(x+1,y-1))
                     ]

        if (x + y) % 2 == 0: neighbors.reverse() 
        results = filter(self.in_bounds, neighbors)  
        return list(results)

class VacuumEnvironment(Grid):
    def __init__(self, rows: int, cols: int):
        super().__init__(rows, cols)
        self.dirty_cells = set()
        self.robot_position = Cell(Position(0,0))
        self.dirty_cost = 1 
        self.total_cost = 0
        self.total_cost_bystep = []

    def add_dirty_cells(self, count: int):
        while len(self.dirty_cells) < count:
            x = random.randint(0, self.cols - 1)
            y = random.randint(0, self.rows - 1)
            self.dirty_cells.add(Cell(Position(x, y)))

    def random_robot_position(self):
        r = random.randint(0, self.rows*self.cols-1)
        return self.cells[r]

    def suck(self, pos): 
        if pos in self.dirty_cells:
            self.dirty_cells.remove(pos)
           
            suck_cost = self.dirty_cost
            self.total_cost += suck_cost
            self.dirty_cost+=1 
            return True, suck_cost
        return False, 0

    def move(self, new_pos):
     
        self.total_cost += 1
        self.dirty_cost += 1   
        self.robot_position = new_pos



def heuristic(a: Cell, b: Cell):
    return max(abs(a.position.x - b.position.x), abs(a.position.y - b.position.y))


def multi_goal_heuristic(current: Cell, goals: Cell):
    if not goals:
        return 0
    return min(heuristic(current, goal) for goal in goals)


def a_star_multigoal(graph, start, goals):
    frontier = PriorityQueue()
    frontier.put((0, start))
    came_from = {} 
    cost_so_far = {}
    came_from[start] = None
    cost_so_far[start] = 0
    
    while not frontier.empty():
        priority_cost, current = frontier.get()

        
        if current in goals:
            break

        for next in graph.neighbors(current): 
            new_cost = cost_so_far[current] + 1 
            if next not in cost_so_far or new_cost < cost_so_far[next]:
                cost_so_far[next] = new_cost 
                priority = new_cost + multi_goal_heuristic(next, goals)
                frontier.put((priority,next)) 
                came_from[next] = current

    return came_from, cost_so_far, current

def reconstruct_path(came_from, start, goal):
    current = goal
    path = []
    if goal not in came_from: 
        return []
    while current != start:
        path.append(current)
        current = came_from[current]
    path.append(start)  
    path.reverse()  
    return path

def vacuum_cleaning_plan(environment: VacuumEnvironment): 
    path = [] 
    current_pos = environment.robot_position


    if current_pos in environment.dirty_cells:
        success, cost = environment.suck(current_pos)
        if success:
            path.append(('suck', current_pos, cost))
            environment.total_cost_bystep.append(environment.total_cost)
          

    while environment.dirty_cells:  
        goals = list(environment.dirty_cells)
        came_from, cost_so_far, reached_goal = a_star_multigoal(
            environment, current_pos, goals)

        
        partial_path = reconstruct_path(came_from, current_pos, reached_goal)
        
        for pos in partial_path[1:]:  
            environment.move(pos)
            path.append(('move', pos))
            
            environment.total_cost_bystep.append(environment.total_cost)

        success, cost = environment.suck(reached_goal)
        if success:
            path.append(('suck', reached_goal, cost))
            environment.total_cost_bystep.append(environment.total_cost)
          
            
        current_pos = reached_goal
        
    return path, environment.total_cost, environment.total_cost_bystep



In [3]:
def get_cleaning_steps(rows: int, cols: int, robot_start: Cell,
                       dirty_positions):

    env = VacuumEnvironment(rows,cols)
    env.robot_position = robot_start
    env.dirty_cells = set(dirty_positions)

    initial_state = {
        'robot_pos': robot_start,
        'dirty_cells': dirty_positions.copy()
    }

    path, total_cost, cost_by_step = vacuum_cleaning_plan(env)
    move_count = sum(1 for a in path if a[0] == 'move')
    suck_count = sum(1 for a in path if a[0] == 'suck')
    
    final_position = env.robot_position

    return {
        'steps': path,
        'total_cost': total_cost,
        'cost_by_step': cost_by_step,
        'move_count': move_count,
        'suck_count': suck_count,
        'initial_state': initial_state,
        'final_position': final_position,
        'rows': rows
    }

In [4]:
def print_steps(result):
    rows = result['rows']  

    print("KẾT QUẢ ROBOT HÚT BỤI")

    print(f"\nTrạng thái ban đầu:")
    robot_pos = result['initial_state']['robot_pos']
    x, y = robot_pos.position.x, robot_pos.position.y

    user_x = x + 1
    user_y = rows - y
    print(f"- Robot tại: ({user_x}, {user_y})")

    dirty_cells = result['initial_state']['dirty_cells']
    print(f"- Các ô bẩn: {[(cell.position.x + 1, rows - cell.position.y) for cell in dirty_cells ]}")
    
    print(f"\nLộ trình chi tiết ({len(result['steps'])} bước - Đã chuẩn hoá về toạ độ người dùng):")
    for i, action in enumerate(result['steps'], 1):
        x, y = action[1].position.x, action[1].position.y
        user_x = x + 1
        user_y = rows - y
        if action[0] == 'move':
            print(f"  Bước {i}: MOVE → ({user_x}, {user_y})| Chi phí tổng: {result['cost_by_step'][i-1]}")
        elif action[0] == 'suck':
            print(f"  Bước {i}: SUCK tại ({user_x}, {user_y}) | Chi phí hút ô dơ: {action[2]} | Chi phí tổng: {result['cost_by_step'][i-1]}")

    print(f"\nThống kê:")
    print(f"- Tổng chi phí: {result['total_cost']}")
    print(f"- Số lần MOVE: {result['move_count']}")
    print(f"- Số lần SUCK: {result['suck_count']}")

    res = result['final_position']
    x, y = res.position.x , res.position.y
    print(f"- Vị trí cuối: ({x + 1}, {rows - y})")

    if result['suck_count'] > 0:
        avg_cost = result['total_cost'] / result['suck_count']
        print(f"  - Chi phí trung bình/ô: {avg_cost:.2f}")

Visualize

In [5]:
import ipywidgets
from ipycanvas import Canvas, hold_canvas
from ipywidgets import HBox, VBox, Layout


In [6]:
class VisualGrid:
    def __init__(self, rows, cols, cell_size= 50):
        self.rows = rows
        self.cols = cols
        self.cell_size = cell_size
        self.canvas = Canvas(width=self.cols*self.cell_size,height=self.rows*self.cell_size)
        self.theme = {
            'grid_bg': "#ffffff",
            'grid_line': '#444',
            "path": "#6a1b9a",
            "text": "#111",
            "robot": "#4ffffc"
            }

    def display(self):
        from IPython.display import display
        display(self.canvas)

    def _fillCell(self, x, y, color, alpha=1):
        x = x * self.cell_size
        y = y * self.cell_size
        if alpha != 1.0:
            self.canvas.global_alpha = alpha
        self.canvas.fill_style = color
        self.canvas.fill_rect(x, y, self.cell_size, self.cell_size)

    def _fillStroke(self, x, y, color='#444', cols=2):
        x = x * self.cell_size
        y = y * self.cell_size
        self.canvas.line_cols = cols
        self.canvas.stroke_style = color
        self.canvas.stroke_rect(x, y, self.cell_size, self.cell_size)

    def _cellText(self, x, y, text, color=None, size=16):
        x = x * self.cell_size
        y = y * self.cell_size
        self.canvas.fill_style = color or self.theme['text']

        self.canvas.font = f'{size}px sans-serif'
        text_x_position = x + self.cell_size/2
        text_y_position = y + self.cell_size/2
        self.canvas.text_align = 'center'
        self.canvas.text_baseline = 'middle'
        self.canvas.fill_text(text, text_x_position, text_y_position)


    def drawRobot(self, r, c):
        self._fillCell(r, c, color='#4ffffc')
        self._fillStroke(r, c)
        self._cellText(r, c, text='R')

    def drawPath(self, path):
        if not path or len(path) < 2:
            return

        for item in path:
            x, y = item.position.x , item.position.y

            self._fillCell(x, y, color="#70b54a", alpha=0.7)
            self._fillStroke(x, y)
            self._cellText(x, y, 'P')

    def drawGrid(self):
        with hold_canvas(canvas=self.canvas):
            self.canvas.clear()
            self.canvas.fill_style = self.theme['grid_bg']
            self.canvas.fill_rect(0, 0, self.cols*self.cell_size, self.rows*self.cell_size)
            self.canvas.stroke_style = self.theme['grid_line']
            self.canvas.line_cols = 2

            for x in range(0, self.cell_size*self.cols, self.cell_size):
                for y in range(0, self.cell_size*self.rows, self.cell_size):
                    self.canvas.stroke_rect(x, y, self.cell_size, self.cell_size)

In [7]:
class GridAnimate:
    def __init__(self, grid, steps):
        self.steps = steps
        self.grid = grid
        self.theme = {
            "clean": "#c8e6c9",
            "frontier": "#e09455",
            "closed": "#b0bec5",
            "dirty": "#f4ff1f",
            "considered": '#ad2457',
            }

    def playComponent(self):
        style = {'description_cols': 'initial'}
        self.slider = ipywidgets.IntSlider(
            min=0, max=len(self.steps)-1,
            description=f'Bước chạy (0-{len(self.steps)-1})',
            style=style,
            layout=Layout(cols='320px', rows='80px')
            )
        return self.slider

    def widgetComponent(self, n):
        if not self.steps:
            return self.grid.canvas
        n = max(0, min(n, len(self.steps)-1))
        self.renderStep(step=self.steps[n])
        return display(self.grid.canvas)

    def Displaying(self):
        controls = self.playComponent()

        out = ipywidgets.interactive_output(self.widgetComponent, {'n': self.slider})
        left = VBox([out], layout=Layout(flex='3 1 0%', cols='auto'))
        right = VBox([controls], layout=Layout(flex='1 1 0%', cols='auto'))

        ui = HBox(
            [left, right],
            layout=Layout(cols='100%', rows='auto', gap='16px')
            )
        return ui

    def renderStep(self, step, show_scores=False):
        self.grid.drawGrid()

        self.dirty = step.get("dirty") or []
        self.cleaned = step.get("cleaned") or []
        self.frontier = step.get("frontier") or []
        self.closed = step.get("closed") or []
        self.considered = step.get("considered") or []
        self.robot = step.get("robot")
        self.path = step.get("path")

        self.g = step.get("g") or {}
        self.f = step.get("f") or {}


        with hold_canvas(self.grid.canvas):

            for item in self.dirty:
                x, y = item.position.x , item.position.y

                self.grid._fillCell(x, y, self.theme["dirty"], alpha=1.0)
                self.grid._fillStroke(x, y)
                self.grid._cellText(x, y, 'D')

            self.grid.drawPath(self.path)

            self.grid.drawRobot(*(self.robot.position.x, self.robot.position.y))

In [8]:
def convert_result_to_viz_steps(result):
    steps = []
    cleaned = []
    dirty_set = set(result['initial_state']['dirty_cells'])

    steps.append({
        "t": 0,
        "action": "init",
        "robot": result['initial_state']['robot_pos'],
        "dirty": sorted(list(dirty_set)),
        "frontier": [],
        "path": [result['initial_state']['robot_pos']],
        "note": "Khởi tạo"
    })


    current_path = [result['initial_state']['robot_pos']]
    for i, step in enumerate(result['steps'], 1):
        action_type = step[0]
        pos = step[1]

        if action_type == 'move':
            current_path.append(pos)
            steps.append({
                "t": i,
                "action": "move",
                "robot": pos,
                "dirty": sorted(list(dirty_set)),
                "frontier": [],
                "path": current_path.copy(),
                "note": f"Di chuyển đến {pos}"
            })
        elif action_type == 'suck':
            if pos in dirty_set:
                dirty_set.remove(pos)
            cleaned.append(pos)
            steps.append({
                "t": i,
                "action": "suck",
                "robot": pos,
                "dirty": sorted(list(dirty_set)),
                "frontier": [],
                "path": current_path.copy(),
                "note": f"Hút bụi tại {pos} - chi phí: {step[2]}"
            })

    return steps

In [9]:
while True:
    try:
        cols = int(input("Nhập số cột: "))
        rows = int(input("Nhập số dòng: "))
        if 1 <= cols and 1 <= rows:
            break
        print("Kích thước phải là số nguyên từ 1 trở lên")
    except ValueError:
        print("Vui lòng nhập số nguyên!")

max_dirty  = cols*rows

while True:
    try:
        dirty_count = int(input(f"Nhập số lượng ô dirty (1-{max_dirty}): "))
        if 1 <= dirty_count <= max_dirty:
            break
    except ValueError:
        print("Vui lòng nhập số nguyên!")

env = VacuumEnvironment(rows, cols)
env.add_dirty_cells(dirty_count)

while True:
    random_start = input("Random vị trí xuất phát của robot (Y/N): ").strip().upper()
    if random_start == 'Y':
        robot_start = env.random_robot_position()
        break
    else:
        start_x = int(input(f"Nhập vị trí x của robot (1-{cols}): "))
        start_y = int(input(f"Nhập vị trí y của robot (1-{rows}): "))
        sys_x = start_x - 1
        sys_y = rows - start_y
        robot_start = Cell(Position(sys_x, sys_y))

        if env.in_bounds(robot_start):
            break
        print(f"Tọa độ robot phải trong khoảng: x(1-{cols}), y(1-{rows})")

dirty_positions = list(env.dirty_cells)
result = get_cleaning_steps(rows, cols, robot_start, dirty_positions)
print_steps(result)


show_viz = input("Hiển thị visualization? (Y/N): ").strip().upper()
if show_viz == 'Y':
    print("Đang khởi tạo visualization...")

    viz_steps = convert_result_to_viz_steps(result)
    visual_grid = VisualGrid(rows, cols)
    animator = GridAnimate(visual_grid, viz_steps)

    print(f"Đã tạo {len(viz_steps)} bước animation")
    print("Sử dụng slider để xem từng bước:")

    from IPython.display import display
    display(animator.Displaying())


KẾT QUẢ ROBOT HÚT BỤI

Trạng thái ban đầu:
- Robot tại: (4, 4)
- Các ô bẩn: [(9, 6), (6, 9), (5, 8), (9, 7), (8, 2)]

Lộ trình chi tiết (18 bước - Đã chuẩn hoá về toạ độ người dùng):
  Bước 1: MOVE → (4, 5)| Chi phí tổng: 1
  Bước 2: MOVE → (5, 6)| Chi phí tổng: 2
  Bước 3: MOVE → (6, 7)| Chi phí tổng: 3
  Bước 4: MOVE → (5, 8)| Chi phí tổng: 4
  Bước 5: SUCK tại (5, 8) | Chi phí hút ô dơ: 5 | Chi phí tổng: 9
  Bước 6: MOVE → (6, 9)| Chi phí tổng: 10
  Bước 7: SUCK tại (6, 9) | Chi phí hút ô dơ: 7 | Chi phí tổng: 17
  Bước 8: MOVE → (7, 8)| Chi phí tổng: 18
  Bước 9: MOVE → (8, 8)| Chi phí tổng: 19
  Bước 10: MOVE → (9, 7)| Chi phí tổng: 20
  Bước 11: SUCK tại (9, 7) | Chi phí hút ô dơ: 11 | Chi phí tổng: 31
  Bước 12: MOVE → (9, 6)| Chi phí tổng: 32
  Bước 13: SUCK tại (9, 6) | Chi phí hút ô dơ: 13 | Chi phí tổng: 45
  Bước 14: MOVE → (10, 5)| Chi phí tổng: 46
  Bước 15: MOVE → (9, 4)| Chi phí tổng: 47
  Bước 16: MOVE → (8, 3)| Chi phí tổng: 48
  Bước 17: MOVE → (8, 2)| Chi phí tổng: 

Trường hợp có thể kiểm tra:

1. Chọn số lượng ô dirty vượt ngoài phạm vi cho phép
2. Chọn vị trí robot
    - random
    - tự chọn
3. Chọn xem visualization:
    - xem
    - không xem
